# Допълнително упражнение 1Б: практика с NumPy


**Машинно самообучение · зимен семестър 2026/2027**

**Repo:** https://github.com/mkorudzhiev/ML_26_27<br>
**Discord:** https://discord.gg/WKztMqX46

Мартин Коруджиев<br/>
mkorudzhiev@uni-sofia.bg<br/>
0885 71 31 60

## Съдържание

1. [Част 1. Форми и избор на данни](#chast-1)
2. [Част 2. Маски и поелементни операции](#chast-2)
3. [Част 3. Оси и broadcasting](#chast-3)
4. [Част 4. Подреждане и разбъркване](#chast-4)

<a id="chast-1"></a>
## Част 1. Форми и избор на данни

### 1. Зареждаме масива

`np.genfromtxt` чете двете числови колони от CSV файла.
`skip_header=1` пропуска заглавния ред. Всеки ред на `homes`
е един имот, а колоните са площ и цена.

In [ ]:
import numpy as np

homes = np.genfromtxt(
    "data/housing_prices.csv", delimiter=",", skip_header=1
)
print("Форма:", homes.shape, "тип:", homes.dtype)
print("Първите два имота:\n", homes[:2])

### 2. Вектор или матрица с една колона

`homes[:, 0]` е едномерен масив с площи. `homes[:, [0]]`
запазва двумерната форма с една колона. И двата избора имат
12 числа, но различна форма. При операции с друг масив
формата определя как се подравняват стойностите.

In [ ]:
areas = homes[:, 0]
prices = homes[:, 1]
area_column = homes[:, [0]]

print("areas:", areas.shape)
print("area_column:", area_column.shape)
print("prices:", prices.shape)
print("Първа площ:", areas[0])

> **Въпрос.** Какви ще са трите форми в този нов пример?
> Кой избор запазва и двете измерения?

```python
import numpy as np
sample = np.arange(20).reshape(5, 4)
print(sample[1:4, 2].shape)
print(sample[1:4, [2]].shape)
print(sample[[1, 3], :2].shape)
```

<details><summary>Отговор</summary>

Формите са `(3,)`, `(3, 1)` и `(2, 2)`. Когато избираме
колона с `[2]`, запазваме двумерния резултат.

</details>

<a id="chast-2"></a>
## Част 2. Маски и поелементни операции

### 3. Едно число за всеки имот

Делим двата едномерни масива поелементно. Резултатът съдържа
цена на квадратен метър за всеки ред. Маската има същата
дължина и избира цели редове от `homes`.

In [ ]:
eur_per_m2 = prices / areas
large_mask = areas >= 80

print("Първите три цени/m²:", np.round(eur_per_m2[:3], 1))
print("Маска:", large_mask)
print("Избрани редове:\n", homes[large_mask])

#### Нека опитаме 1

Създаваме `selected_homes` с имотите, които са **поне 80 m²**
и струват **най-много 220000 евро**. Комбинираме две маски с `&`,
като ограждаме всяко сравнение в скоби. Избираме цели редове;
очакваме форма `(2, 2)`.

In [ ]:
# TODO: selected_homes = ...

In [ ]:
if "selected_homes" in globals():
    assert selected_homes.shape == (2, 2)
    assert np.array_equal(selected_homes[:, 0], [90, 85])
    assert np.array_equal(selected_homes[:, 1], [210_000, 205_000])
    print("Вярно:\n", selected_homes)
else:
    print("Попълваме избора и изпълняваме проверката пак.")

<details><summary>Един възможен отговор</summary>

```python
selected_homes = homes[(areas >= 80) & (prices <= 220_000)]
```

Скобите около сравненията са важни за реда на операциите.

</details>

> **Въпрос.** Какво ще отпечата кодът? Защо `and` не е подходящ
> за две NumPy маски?

```python
x = np.array([4, 7, 10, 13])
print(x[(x > 5) & (x % 2 == 0)])
```

<details><summary>Отговор</summary>

Получаваме `[10]`. `&` съчетава условията поелементно.
`and` очаква една булева стойност за целия масив и тук
повдига `ValueError`.

</details>

<a id="chast-3"></a>
## Част 3. Оси и broadcasting

### 4. Обобщаваме по колони

При `axis=0` извършваме операцията **по колони**: обхождаме стойностите надолу по редовете и изчисляваме по един резултат за всяка колона.

Ако данните имат две колони (**площ** и **цена** в нашия пример), резултатът ще съдържа две числа. Първото е изчислено от всички стойности в колоната за площ, а второто - от всички стойности в колоната за цена.

Важно е тези числа да се разглеждат отделно, защото описват различни величини и имат различни мерни единици. Например едното е в **квадратни метри**, а другото - в **лева**.

In [ ]:
column_means = homes.mean(axis=0)
column_mins = homes.min(axis=0)
column_maxs = homes.max(axis=0)

print("Средни:", column_means, "форма:", column_means.shape)
print("Минимуми:", column_mins)
print("Максимуми:", column_maxs)

#### Нека опитаме 2

Намираме `column_ranges`: за всяка колона изваждаме минимума
от максимума. После създаваме `centered_homes`, като извадим
средната на всяка колона от съответната колона. Проверяваме
формите и че средните на новите колони са приблизително нула.

Това е упражнение за оси и broadcasting. Ако подготвяме данни за
модел, статистиките се намират само от тренировъчните редове.

In [ ]:
# TODO: column_ranges = ...
# TODO: centered_homes = ...

In [ ]:
if "column_ranges" in globals() and "centered_homes" in globals():
    assert np.array_equal(column_ranges, [75, 150_000])
    assert centered_homes.shape == homes.shape
    assert np.allclose(centered_homes.mean(axis=0), [0, 0])
    print("Вярно: размах", column_ranges)
    print("Средни след центриране:", centered_homes.mean(axis=0))
else:
    print("Попълваме двете стойности и изпълняваме проверката пак.")

<details><summary>Един възможен отговор</summary>

```python
column_ranges = homes.max(axis=0) - homes.min(axis=0)
centered_homes = homes - homes.mean(axis=0)
```

`homes` има форма `(12, 2)`, а средните имат форма `(2,)`.
NumPy изважда всяка средна от нейната колона.

</details>

### 5. Създаване на нова ос

`prices[:, None]` превръща формата `(12,)` в `(12, 1)`.
Умножаваме я по три коефициента с форма `(3,)`. Резултатът има 12 реда и три колони: намалена, оригинална и увеличена
цена за всеки имот.

Нова ос е нужна, за да кажем на NumPy, че искаме всяка от 12-те цени да се комбинира с всеки от трите коефициента.
Без новата ос prices има форма (12,) - това е едномерен масив. След prices[:, None] формата става (12, 1): 12 реда и една колона. Тогава NumPy може чрез broadcasting да комбинира тази колона с масив с форма (3,), който се разглежда като три стойности по колоните. Така формите (12, 1) и (3,) се разширяват до (12, 3).

С други думи, новата ос не добавя нови данни. Тя само променя структурата на масива, така че NumPy да разбере, че цените трябва да бъдат разположени по редовете, а трите коефициента - по колоните.

In [ ]:
factors = np.array([0.95, 1.00, 1.05])
scenarios = prices[:, None] * factors

print("Форми:", prices[:, None].shape, factors.shape, scenarios.shape)
print("Първият имот:", scenarios[0])

#### Нека опитаме 3

Създаваме `adjusted_per_m2` с цената на квадратен метър за
всеки имот и за трите сценария. Резултатът трябва да има
форма `(12, 3)`. Използваме `scenarios` и `areas`, без цикъл.
Преди проверката решаваме коя форма на `areas` е нужна.

In [ ]:
# TODO: adjusted_per_m2 = ...

In [ ]:
if "adjusted_per_m2" in globals():
    assert adjusted_per_m2.shape == (12, 3)
    assert np.allclose(adjusted_per_m2[:, 1], eur_per_m2)
    assert np.isclose(adjusted_per_m2[0, 0], 110_000 * 0.95 / 40)
    print("Вярно: първият имот", adjusted_per_m2[0])
else:
    print("Попълваме масива и изпълняваме проверката пак.")

<details><summary>Един възможен отговор</summary>

```python
adjusted_per_m2 = scenarios / areas[:, None]
```

При форма `(12, 1)` всяка площ се дели от трите цени
на съответния ред.

</details>

<a id="chast-4"></a>
## Част 4. Подреждане и разбъркване

### 6. Намираме реда на най-ниските цени на квадратен метър

`np.argsort` връща индексите в ред на нарастване на
`eur_per_m2`. С тези индекси избираме цели редове от `homes`,
за да запазим площ и цена заедно.

In [ ]:
order = np.argsort(eur_per_m2)
cheapest_three = homes[order[:3]]

print("Индекси:", order[:3])
print("Имотите:\n", cheapest_three)
print("Цени/m²:", np.round(eur_per_m2[order[:3]], 1))

### 7. Разбъркваме редовете възпроизводимо

`default_rng(seed)` създава генератор. `permutation` връща
всички индекси точно по веднъж в нов ред. Едно и също `seed`
дава един и същ ред. Прилагаме индексите към **целите редове**,
за да не разменим цената на един имот с площта на друг.

In [ ]:
rng = np.random.default_rng(seed=17)
shuffled_indices = rng.permutation(len(homes))
shuffled_homes = homes[shuffled_indices]

print("Първи индекси:", shuffled_indices[:5])
print("Първи разбъркан имот:", shuffled_homes[0])
print("Същият оригинален ред:", homes[shuffled_indices[0]])

#### Нека опитаме 4

Пишем `shuffle_rows(data, seed)`, която връща нов масив
с разбъркани **цели редове**. Два пъти с един и същ `seed`
трябва да получим еднакъв резултат. Използваме `default_rng`
и `permutation`. Не променяме входния масив.

In [ ]:
# TODO: Дописваме функцията.
# def shuffle_rows(data, seed):
#     ...

In [ ]:
if "shuffle_rows" in globals():
    original_homes = homes.copy()
    first = shuffle_rows(homes, 42)
    second = shuffle_rows(homes, 42)
    assert first.shape == homes.shape
    assert np.array_equal(first, second)
    assert not np.array_equal(first, homes)
    assert np.array_equal(
        np.sort(first[:, 0]), np.sort(homes[:, 0])
    )
    assert all(
        np.any(np.all(homes == row, axis=1)) for row in first
    )
    assert np.array_equal(homes, original_homes)
    print("Вярно: редовете са запазени и редът е възпроизводим.")
else:
    print("Попълваме функцията и изпълняваме проверката пак.")

<details><summary>Един възможен отговор</summary>

```python
def shuffle_rows(data, seed):
    rng = np.random.default_rng(seed)
    indices = rng.permutation(len(data))
    return data[indices]
```

Изборът с масив от индекси връща нов масив с цели редове.

</details>